In [3]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from xgboost import XGBClassifier
import joblib

In [4]:
df = pd.read_csv("../data/processed/cleaned_data.csv")

X = df.drop(columns=["addicted_label"])
y = df["addicted_label"]

In [5]:
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.30, random_state=42, stratify=y
)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.50, random_state=42, stratify=y_temp
)

print(f"Train: {X_train.shape}, Val: {X_val.shape}, Test: {X_test.shape}")

Train: (483958, 15), Val: (103705, 15), Test: (103706, 15)


In [6]:
param_grid = {
    "n_estimators": [100, 200, 300],
    "max_depth": [3, 5, 7],
    "learning_rate": [0.01, 0.1, 0.2],
    "subsample": [0.8, 1.0],
    "colsample_bytree": [0.8, 1.0],
}

xgb = XGBClassifier(
    objective="binary:logistic",
    eval_metric="logloss",
    random_state=42,
)   

In [7]:
grid_search = GridSearchCV(
    estimator=xgb,
    param_grid=param_grid,
    scoring="accuracy",
    cv=5,
    n_jobs=-1,
    verbose=2,
)

grid_search.fit(X_train, y_train)

print("Best params:", grid_search.best_params_)
print("Best CV score:", grid_search.best_score_)

Fitting 5 folds for each of 108 candidates, totalling 540 fits
Best params: {'colsample_bytree': 0.8, 'learning_rate': 0.2, 'max_depth': 7, 'n_estimators': 300, 'subsample': 1.0}
Best CV score: 0.9326077916717409


In [8]:
best_model = grid_search.best_estimator_

val_preds = best_model.predict(X_val)
print("Validation Accuracy:", accuracy_score(y_val, val_preds))
print(classification_report(y_val, val_preds))

Validation Accuracy: 0.9328479822573646
              precision    recall  f1-score   support

           0       0.89      0.87      0.88     30134
           1       0.95      0.96      0.95     73571

    accuracy                           0.93    103705
   macro avg       0.92      0.91      0.92    103705
weighted avg       0.93      0.93      0.93    103705



In [9]:
test_preds = best_model.predict(X_test)
print("Test Accuracy:", accuracy_score(y_test, test_preds))
print(classification_report(y_test, test_preds))
print(confusion_matrix(y_test, test_preds))

Test Accuracy: 0.9325786357587796
              precision    recall  f1-score   support

           0       0.89      0.87      0.88     30135
           1       0.95      0.96      0.95     73571

    accuracy                           0.93    103706
   macro avg       0.92      0.91      0.92    103706
weighted avg       0.93      0.93      0.93    103706

[[26298  3837]
 [ 3155 70416]]


In [10]:
import joblib

joblib.dump(best_model, "../models/xgb_addiction_model.pkl")
print("Model saved to ../models/xgb_addiction_model.pkl")

Model saved to ../models/xgb_addiction_model.pkl
